# Lab 02 - Clusters y Optimización

**Objetivos:**
- Probar autoscaling con diferentes cargas
- Calcular costos de diferentes configuraciones
- Optimizar configuración de clusters

## Ejercicio 1: Test de Autoscaling - Carga Ligera

In [ ]:
from pyspark.sql.functions import rand
import time

print("🔹 Carga Ligera: Procesando 1 millón de registros")
start = time.time()

# Crear DataFrame pequeño
df_light = spark.range(0, 1_000_000) \
    .withColumn("value", rand() * 1000) \
    .withColumn("category", (rand() * 10).cast("int"))

# Agregación simple
result = df_light.groupBy("category").count().collect()

end = time.time()
print(f"✅ Completado en {end - start:.2f} segundos")
print(f"📊 Resultados: {len(result)} categorías")

## Ejercicio 2: Test de Autoscaling - Carga Pesada

In [ ]:
print("🔸 Carga Pesada: Procesando 50 millones de registros")
start = time.time()

# Crear DataFrame grande con repartición
df_heavy = spark.range(0, 50_000_000) \
    .withColumn("value", rand() * 1000) \
    .withColumn("category", (rand() * 100).cast("int")) \
    .repartition(16)  # Forzar más paralelismo

# Agregación compleja
from pyspark.sql.functions import avg, sum, stddev, min, max

result = df_heavy.groupBy("category").agg(
    avg("value").alias("avg_value"),
    sum("value").alias("sum_value"),
    stddev("value").alias("stddev_value"),
    min("value").alias("min_value"),
    max("value").alias("max_value")
).collect()

end = time.time()
print(f"✅ Completado en {end - start:.2f} segundos")
print(f"📊 Resultados: {len(result)} categorías con estadísticas")
print(f"\n💡 Revisa la UI de Spark para ver el autoscaling en acción")

## Ejercicio 3: Calculadora de Costos

In [ ]:
def calcular_costo_cluster(vm_type, num_workers, horas_mes, tipo_cluster="all_purpose", usar_spot=False):
    """
    Calcula el costo mensual de un cluster de Databricks
    """
    # Configuración de precios (ejemplo para East US)
    vm_precios = {
        "Standard_DS3_v2": {"cores": 4, "ram_gb": 14, "precio_hora": 0.192},
        "Standard_DS4_v2": {"cores": 8, "ram_gb": 28, "precio_hora": 0.384},
        "Standard_DS5_v2": {"cores": 16, "ram_gb": 56, "precio_hora": 0.768},
        "Standard_E4s_v3": {"cores": 4, "ram_gb": 32, "precio_hora": 0.252}
    }
    
    # DBU pricing (Premium tier)
    dbu_precios = {
        "all_purpose": 0.55,  # USD por DBU
        "jobs": 0.22          # USD por DBU (50% más barato)
    }
    
    vm_config = vm_precios.get(vm_type, vm_precios["Standard_DS3_v2"])
    
    # Calcular DBUs (aproximado: cores * 0.75)
    dbus_por_hora = vm_config["cores"] * 0.75 * num_workers
    
    # Costos por hora
    costo_vm_hora = vm_config["precio_hora"] * num_workers
    costo_dbu_hora = dbus_por_hora * dbu_precios[tipo_cluster]
    
    # Aplicar descuento spot
    if usar_spot:
        costo_vm_hora *= 0.60  # 40% descuento promedio
    
    costo_total_hora = costo_vm_hora + costo_dbu_hora
    costo_mensual = costo_total_hora * horas_mes
    
    return {
        "vm_type": vm_type,
        "workers": num_workers,
        "cores_total": vm_config["cores"] * num_workers,
        "ram_total_gb": vm_config["ram_gb"] * num_workers,
        "dbus_hora": round(dbus_por_hora, 2),
        "costo_vm_hora": round(costo_vm_hora, 2),
        "costo_dbu_hora": round(costo_dbu_hora, 2),
        "costo_total_hora": round(costo_total_hora, 2),
        "costo_mensual": round(costo_mensual, 2),
        "tipo_cluster": tipo_cluster,
        "usa_spot": usar_spot
    }

# Ejemplo de uso
print("💰 Calculadora de Costos de Cluster\n")

In [ ]:
# Escenario 1: Desarrollo - All-Purpose con autoscaling
dev_config = calcular_costo_cluster(
    vm_type="Standard_DS3_v2",
    num_workers=3,  # Promedio entre 2-4
    horas_mes=160,  # 8 hrs/día * 20 días
    tipo_cluster="all_purpose"
)

print("📊 Escenario 1: Desarrollo (All-Purpose)")
print(f"  VM: {dev_config['vm_type']}")
print(f"  Workers: {dev_config['workers']}")
print(f"  Cores: {dev_config['cores_total']}")
print(f"  RAM: {dev_config['ram_total_gb']} GB")
print(f"  DBUs/hora: {dev_config['dbus_hora']}")
print(f"  💵 Costo/hora: ${dev_config['costo_total_hora']}")
print(f"  💵 Costo/mes: ${dev_config['costo_mensual']}")
print()

In [ ]:
# Escenario 2: Producción - Job Cluster con Spot
prod_config = calcular_costo_cluster(
    vm_type="Standard_DS4_v2",
    num_workers=6,
    horas_mes=180,  # Jobs nocturnos ~6 hrs/día
    tipo_cluster="jobs",
    usar_spot=True
)

print("📊 Escenario 2: Producción (Job + Spot)")
print(f"  VM: {prod_config['vm_type']}")
print(f"  Workers: {prod_config['workers']} (con Spot Instances)")
print(f"  Cores: {prod_config['cores_total']}")
print(f"  RAM: {prod_config['ram_total_gb']} GB")
print(f"  DBUs/hora: {prod_config['dbus_hora']}")
print(f"  💵 Costo/hora: ${prod_config['costo_total_hora']}")
print(f"  💵 Costo/mes: ${prod_config['costo_mensual']}")
print()

# Comparación
ahorro = dev_config['costo_mensual'] - prod_config['costo_mensual']
porcentaje = (ahorro / dev_config['costo_mensual']) * 100

print(f"💡 Ahorro con Job + Spot vs All-Purpose: ${ahorro:.2f}/mes ({porcentaje:.1f}%)")

## ✅ Lab Completado

Has aprendido:
- ✅ Probar autoscaling con diferentes cargas
- ✅ Calcular costos de clusters
- ✅ Comparar configuraciones